In [ ]:
import xarray as xr
import pandas as pd
import numpy as np

# Load the dataset once
ds = xr.open_dataset('data/VerticalVelocity.nc', engine="netcdf4")

# Convert time to datetime
ds = ds.assign_coords(date=pd.to_datetime(ds['valid_time'].values))

# Create list to collect anomaly DataArrays
anomaly_list = []

# Loop through each pressure level
for p in ds['pressure_level'].values:
    # Select current pressure level
    dsh = ds.sel(pressure_level=p).squeeze()

    # Convert to DataFrame
    df = dsh['w'].to_dataframe().reset_index()

    # Extract year and month
    df['year'] = df['valid_time'].dt.year
    df['month'] = df['valid_time'].dt.month

    # Assign season
    df['season'] = np.select(
        [
            df['month'].isin([12, 1, 2]),
            df['month'].isin([3, 4, 5]),
            df['month'].isin([6, 7, 8]),
            df['month'].isin([9, 10, 11])
        ],
        ['DJF', 'MAM', 'JJA', 'SON']
    )

    # Adjust season year: Dec belongs to next year
    df['season_year'] = df['year']
    df.loc[df['month'] == 12, 'season_year'] += 1

    # STEP 1: Seasonal mean for each year
    seasonal = df.groupby(['latitude', 'longitude', 'season_year', 'season'])['w'].mean().reset_index()

    # STEP 2: Climatology (mean over years per season)
    clim = seasonal.groupby(['latitude', 'longitude', 'season'])['w'].mean().reset_index()
    clim.rename(columns={'w': 'w_clim'}, inplace=True)

    # STEP 3: Merge and compute anomaly
    merged = pd.merge(seasonal, clim, on=['latitude', 'longitude', 'season'], how='left')
    merged['w_anomaly'] = merged['w'] - merged['w_clim']

    # STEP 4: Reconstruct time index
    merged['date_str'] = merged['season'] + ' ' + merged['season_year'].astype(str)
    
    # Approximate center date for each season (optional refinement)
    season_to_month = {'DJF': 1, 'MAM': 4, 'JJA': 7, 'SON': 10}
    merged['month'] = merged['season'].map(season_to_month)
    merged['day'] = 15
    merged['valid_time'] = pd.to_datetime({
        'year': merged['season_year'],
        'month': merged['month'],
        'day': merged['day']
    })

    # Pivot back to xarray format
    da = merged.set_index(['valid_time', 'latitude', 'longitude'])[['w_anomaly']].to_xarray()['w_anomaly']
    
    # Add back pressure level as new dimension
    da = da.expand_dims(pressure_level=[p])

    # Append to list
    anomaly_list.append(da)

# Merge all levels into one xarray
anomaly_all_levels = xr.concat(anomaly_list, dim='pressure_level')

# Sort pressure levels if needed
anomaly_all_levels = anomaly_all_levels.sortby('pressure_level')

# Wrap into Dataset
anomaly_ds = xr.Dataset({'w_anomaly': anomaly_all_levels})

In [ ]:
anomaly_ds.to_netcdf("VerticalVelocity_SeasonalAnomalies_AllLevels.nc")